In [ ]:
from collections import defaultdict
import pandas as pd

pathways = pd.read_csv('trip_pathways.csv')
trip_graph = defaultdict(dict)

for idx, row in pathways.iterrows():
    trip_graph[row['start_trip_id']][row['end_trip_id']] = idx 

pathways_dict = pathways.to_dict('index')
print(f"Graph built with {len(trip_graph)} unique starting trips and {sum(len(v) for v in trip_graph.values())} edges.")


In [ ]:
from collections import deque, defaultdict


def find_journeys_complex_cost(graph, pathways_dict, start_trips, goal_trips, max_transfers,traffic,
    get_transport_time, # function
    get_cost             # function
):
    results = []
    # (current_trip_id, start_stop_id, path_list, cumulative_costs_to_node)
    queue = deque()
    best_costs_to_node = defaultdict(lambda: {
        'money': float('inf'),
        'transport_time': float('inf'),
        'walk': float('inf')
    })
    # queue with all start trips
    for start_trip_id, data in start_trips.items():
        # cost of starting firest trip
        costs = {
            'money': 0,
            'transport_time': 0,
            'walk': data['walk']
        }
        path = [start_trip_id]
        # queue should have (trip_id, start_stop, path, costs)
        queue.append((start_trip_id, data['start_stop'], path, costs))

        best_costs_to_node[start_trip_id] = costs.copy()
        # (0 transfers)
        if start_trip_id in goal_trips:
            final_walk = goal_trips[start_trip_id]['walk']
            final_journey_costs = costs.copy()
            final_journey_costs['walk'] += final_walk
            results.append((path, final_journey_costs))
    # bfs
    while queue:
        (current_trip, start_stop, path, current_costs) = queue.popleft()
        num_transfers = len(path) - 1
        if num_transfers >= max_transfers:
            continue
        # expand
        for next_trip, pathway_id in graph.get(current_trip, {}).items():
            pathway = pathways_dict[pathway_id]
            if next_trip in path:
                continue
            # 1. Cost of the transfer walk
            transfer_walk_cost = pathway['walking_distance_m']
            # 2. Cost till the next trip
            next_trip_money = get_cost(
                current_trip, 
                start_stop,
                pathway['start_stop_id']
                # now i need transfer_info to have start_stop, end_stop, walking dist (i have it in pathways tho)
                # current_trip should have a start_stop
            )
            next_trip_time = get_transport_time(
                traffic, 
                current_trip, 
                start_stop, 
                pathway['start_stop_id']
            )
            new_costs = {
                'money': current_costs['money'] + next_trip_money,
                'transport_time': current_costs['transport_time'] + next_trip_time,
                'walk': current_costs['walk'] + transfer_walk_cost
            }
            # prune
            best_known = best_costs_to_node[next_trip]
            is_better = (
                new_costs['money'] < best_known['money'] or
                new_costs['transport_time'] < best_known['transport_time'] or
                new_costs['walk'] < best_known['walk']
            )
            if is_better:
                best_costs_to_node[next_trip]['money'] = min(
                    best_known['money'], new_costs['money']
                )
                best_costs_to_node[next_trip]['transport_time'] = min(
                    best_known['transport_time'], new_costs['transport_time']
                )
                best_costs_to_node[next_trip]['walk'] = min(
                    best_known['walk'], new_costs['walk']
                )
                # Add to the queue to explore further
                new_path = path + [next_trip]

                queue.append((next_trip, pathway['end_stop_id'], new_path, new_costs))
                # k transfers goal?
                if next_trip in goal_trips:
                    final_walk = goal_trips[next_trip]['walk']
                    final_journey_costs = new_costs.copy()
                    final_journey_costs['walk'] += final_walk
                    results.append((new_path, final_journey_costs))

    return results